# Extracting Coolattin townlands and measurements from `townlands.json`

This notebook shows exactly how the uploaded GeoJSON file was turned into a table of **townland names** and **measurements**.

## Key point
Even though the file looks like raw geometry, each feature has a `properties` object that contains the townland metadata, including:

- `TL_ENGLISH` → English townland name
- `TL_GAEILGE` → Irish townland name
- `Shape__Area` → area in square metres
- `Shape__Length` → perimeter / boundary length in metres
- `COUNTY_ENGLISH` → county name

So the workflow is:

1. Load the JSON
2. Iterate through `features`
3. Read each feature's `properties`
4. Extract the relevant fields
5. Convert area into hectares and acres
6. Build a flat table for review/export


In [ ]:
import json
from pathlib import Path
import pandas as pd

json_path = Path('/mnt/data/townlands.json')

with json_path.open('r', encoding='utf-8') as f:
    data = json.load(f)

type(data), data.keys(), len(data['features'])

## Inspect one feature

A GeoJSON feature has two main parts:

- `geometry` → polygon coordinates
- `properties` → the useful descriptive fields

The names were extracted from `properties`, not from the geometry coordinates.


In [ ]:
first_feature = data['features'][0]
first_feature.keys()

In [ ]:
properties = first_feature['properties']
sorted(properties.keys())[:20], len(properties.keys())

In [ ]:
properties

## Extract the fields we care about

Below, each row is built from the `properties` of one feature.


In [ ]:
rows = []

for feature in data['features']:
    props = feature.get('properties', {})
    row = {
        'feature_id': feature.get('id'),
        'object_id': props.get('OBJECTID'),
        'townland_id': props.get('TD_ID'),
        'townland_english': props.get('TL_ENGLISH'),
        'townland_irish': props.get('TL_GAEILGE'),
        'county': props.get('COUNTY_ENGLISH'),
        'area_m2': props.get('Shape__Area') if props.get('Shape__Area') is not None else props.get('AREA'),
        'perimeter_m': props.get('Shape__Length') if props.get('Shape__Length') is not None else props.get('Shape__Len'),
        'population_1827': props.get('T_POP_1827'),
        'population_1839': props.get('T_POP_1839_'),
        'population_1848': props.get('T_POP_1848'),
        'population_1850': props.get('T_POP_1850'),
        'population_1860': props.get('T_POP_1860'),
        'population_1868': props.get('T_POP_1868'),
        'total_clearances': props.get('Total_Clearances'),
    }
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

## Add converted measurements

The raw file stores area in square metres.  
We can derive:

- **hectares** = square metres / 10,000
- **acres** = square metres / 4,046.8564224


In [ ]:
df['area_hectares'] = df['area_m2'] / 10_000
df['area_acres'] = df['area_m2'] / 4046.8564224

df[['townland_english', 'area_m2', 'area_hectares', 'area_acres', 'perimeter_m']].head(10)

## Why the count was different from the earlier rough estimate

The earlier rough estimate was based on a truncated preview of the JSON, not the full file.

From the full file:
- there are **152 features**
- there are **150 unique English townland names**
- 2 names appear twice, which is why raw feature count and unique-name count differ


In [ ]:
feature_count = len(df)
unique_name_count = df['townland_english'].nunique(dropna=True)
duplicate_name_counts = (
    df['townland_english']
    .value_counts(dropna=True)
    .loc[lambda s: s > 1]
    .rename_axis('townland_english')
    .reset_index(name='count')
)

feature_count, unique_name_count, duplicate_name_counts

## Deduplicated townland table

If you want one row per townland name, you can deduplicate by `townland_english`.


In [ ]:
townlands_dedup = (
    df.sort_values(['townland_english', 'object_id'])
      .drop_duplicates(subset=['townland_english'], keep='first')
      .reset_index(drop=True)
)

townlands_dedup[['townland_english', 'townland_irish', 'area_m2', 'area_hectares', 'area_acres', 'perimeter_m']].head(15)

## Export to Excel

This is the same basic idea used to create the spreadsheet.


In [ ]:
output_path = Path('/mnt/data/townlands_extracted_from_json_demo.xlsx')

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    summary = pd.DataFrame([
        {'metric': 'feature_count', 'value': feature_count},
        {'metric': 'unique_townland_names', 'value': unique_name_count},
    ])
    summary.to_excel(writer, sheet_name='Summary', index=False)
    df.to_excel(writer, sheet_name='Raw_Features', index=False)
    townlands_dedup.to_excel(writer, sheet_name='Deduplicated_Townlands', index=False)

output_path.as_posix()

## Bottom line

You could not easily see the names at first because the file preview was dominated by the `geometry.coordinates` arrays.

The names and measurements were inside:

```python
feature['properties']['TL_ENGLISH']
feature['properties']['Shape__Area']
feature['properties']['Shape__Length']
```

That `properties` block is where the townland metadata lives.
